# Polaris Privilege — Real-World Use-Case Role Test

Validates the four production role profiles from `privilege/doc-privilege-test.md` §3 by
provisioning each as a **multi-privilege worker** and asserting both sides of its boundary:
every capability the role is meant to have must classify `AUTHORIZED`, and every action it
must be denied must classify `BLOCKED_PRIV`.

| Role | Footprint | Intent: must do | Intent: blocked from |
|---|---|---|---|
| **A · Data Engineer** | `NAMESPACE_CREATE, TABLE_CREATE, TABLE_WRITE_DATA, TABLE_READ_DATA, VIEW_CREATE` | build & maintain datasets | drop tables / drop namespaces |
| **B · BI Analyst** | `TABLE_LIST, TABLE_READ_DATA, VIEW_LIST, VIEW_CREATE` | read tables, build views | create/modify/drop tables |
| **C · Auditor** | `CATALOG_MANAGE_METADATA` | list/inspect all metadata | mutate structure |
| **D · Tenant Admin** | `CATALOG_MANAGE_CONTENT` | full control of *its* catalog | other catalogs / mgmt layer |

The table above is the role **intent**. The pass/fail assertions in this notebook use the
**measured baseline** (`doc-privilege-results.md` §4, 2026-07-02), so a green run confirms the
build still behaves as recorded. Where measured ≠ intent, the fix is a footprint change
(results §7), noted inline — specifically: **B**'s `VIEW_LIST` does not authorize reading a
single view (`View.GET → 403`; needs `VIEW_READ_PROPERTIES`), and **C**'s
`CATALOG_MANAGE_METADATA` is not read-only (it authorizes every mutation), so the CMM Auditor
has no deny boundary here. Each action runs in its own freshly-scaffolded catalog, so role
tests never contaminate each other.

> **Scope caveat (Auditor):** the doc says `CATALOG_MANAGE_METADATA` exposes metadata but
> hides *raw rows* (`TABLE_READ_DATA` absent). Raw-row access in Polaris is gated at
> credential-vending / object storage, **not** the Iceberg REST catalog API this harness
> drives — so that specific boundary is out of scope here and noted rather than asserted.

**Case structure** — each assertion is one test case composed by `run_case` (in `src`):
**Suite** (`build_suite`, common) arranges a catalog + scaffold + a multi-privilege worker and
*records* what it creates → **Execute** (per-action) fires the challenge → **Result** (`classify`,
pure) derives the outcome → **Cleaner** (`clean`, common) deletes the recorded entities in a
`finally`. The cross-catalog test supplies its own two-catalog Suite but reuses the same
orchestrator, so both catalogs + the worker are always reclaimed.

In [52]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import time, uuid
import importlib, nb_support
importlib.reload(nb_support)
from nb_support import *
init_env("local")                       # default safe target; switch to "dev" for shared cluster
require_not_prod("privilege hierarchy / role mutations")  # hard-fail against company PROD

# ── global request timeout: no single HTTP call may hang the notebook ────────
# Patches the shared requests.Session so EVERY call (notebook + src utils) gets a
# (connect, read) timeout. A dead/slow endpoint now raises promptly instead of
# blocking forever — probe() catches it and records ERROR.
import requests as _rq
if not getattr(_rq.sessions.Session, "_timeout_patched", False):
    _orig_request = _rq.sessions.Session.request
    def _request_with_timeout(self, method, url, **kw):
        kw.setdefault("timeout", (5, 30))          # 5s connect, 30s read
        return _orig_request(self, method, url, **kw)
    _rq.sessions.Session.request = _request_with_timeout
    _rq.sessions.Session._timeout_patched = True

tok = root_token()
_RAW_SETTLE = 0.6                        # settle after a read-after-write lag-500
NS = lambda cat: f"ns-{cat}"            # per-catalog namespace (Vector B: no flat strings)
print("✅ privilege harness loaded")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ privilege harness loaded


In [53]:
# ── action registry + entity bodies ──────────────────────────────────────────
# Each action: build(cat,hdr,ctx) scaffolds the prereq AS THE SCAFF principal;
# challenge(cat,hdr,ctx) fires the action under test AS THE WORKER.
def _tbl_body(cat, ns, tbl):
    return {"name": tbl, "location": f"s3a://{BUCKET}/{cat}/{ns}/{tbl}/",
            "schema": {"type":"struct","fields":[
                {"id":1,"name":"id","type":"long","required":True},
                {"id":2,"name":"value","type":"string","required":False}]}}

def _view_body(cat, ns, vw):
    return {"name": vw, "default-namespace":[ns],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),
                            "schema-id":0,"default-namespace":[ns],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{ns}/{vw}/"}

def _mk_ns(cat, hdr):
    requests.post(f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
                  json={"namespace":[NS(cat)], "properties":{}})
def _mk_tbl(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
                  json=_tbl_body(cat, NS(cat), ctx["entity"]))
def _mk_view(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
                  json=_view_body(cat, NS(cat), ctx["entity"]))

# action_id -> spec.  drop=True actions get the config gate neutralized (gap 1).
ACTIONS = {
  "ns_create":  dict(target="Namespace", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: None,
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
         json={"namespace":[NS(cat)], "properties":{}})),
  "ns_list":    dict(target="Namespace", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr)),
  "ns_drop":    dict(target="Namespace", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}", headers=hdr)),
  "tbl_create": dict(target="Table", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
         json=_tbl_body(cat, NS(cat), ctx["entity"]))),
  "tbl_list":   dict(target="Table", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr)),
  "tbl_read":   dict(target="Table", action="READ", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "tbl_commit": dict(target="Table", action="COMMIT", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr,
         json={"requirements":[], "updates":[
             {"action":"set-properties","updates":{"probe":"1"}}]})),
  "tbl_drop":   dict(target="Table", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "view_create":dict(target="View", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
         json=_view_body(cat, NS(cat), ctx["entity"]))),
  "view_get":   dict(target="View", action="GET", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
  "view_drop":  dict(target="View", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
}
LABEL = {k: f"{v['target']}.{v['action']}" for k,v in ACTIONS.items()}

# ── six-way classifier (identical taxonomy to the matrix notebook) ───────────
def _is_lag(r):
    b = r.text.lower()
    return r.status_code == 500 and ("metadata" in b or "nullpointer" in b)

def classify(r):
    sc, b = r.status_code, r.text.lower()
    if sc in (200, 201, 204):                        return "AUTHORIZED"
    if sc == 409 or "already exists" in b:           return "AUTHORIZED"   # create lag-committed
    if _is_lag(r):                                   return "LAG_500"
    if sc == 403:
        if "unable to purge" in b:                   return "BLOCKED_CONFIG"  # harness bug
        if "is not authorized" in b or "not authorized for op" in b: return "BLOCKED_PRIV"
        if "opa" in b or "ext_authz" in b or "denied by policy" in b: return "BLOCKED_OPA"  # harness bug
        return "BLOCKED_PRIV"
    if sc == 404:                                    return "NOT_FOUND_404"   # setup issue
    return f"HTTP_{sc}"

# ── one probe = one test case, composed via run_case (from src) ──────────────
# The four responsibilities (arrange / act / assert / cleanup):
#   Suite   = build_suite(...)   COMMON  — fresh catalog + scaff prereq + worker
#   Execute = spec["challenge"]  PER-CELL — fire the action under test
#   Result  = classify(...)      PURE    — derive the outcome, change nothing
#   Cleaner = clean(...)         COMMON  — reclaim every tracked entity, ALWAYS
# run_case wires them with a try/finally so cleanup is guaranteed, and cleanup
# deletes the *recorded* Entities (not a naming guess) — no probe can leak.
def probe(action_id, privileges, verbose=False):
    """Returns (outcome, granted, invalid). All resources are reclaimed before it
    returns — even on exception — because run_case cleans in a `finally`."""
    spec = ACTIONS[action_id]
    privs = list(privileges) if isinstance(privileges, (list, tuple)) else [privileges]

    def suite(ents, root):                               # 1. arrange (common)
        build_suite(ents, root, prefix=f"hzr-{action_id.replace('_','')}-",
                    build=lambda cat, hdr, ctx: spec["build"](cat, hdr, ctx),
                    footprint=privs, drop_gate=spec["drop"])

    def execute(suite_ents, created, root):              # 2. act (per-cell)
        c = suite_ents.ctx
        if c.get("worker_hdr") is None:
            return None                                  # token not issued
        fire = lambda: spec["challenge"](c["catalog"], c["worker_hdr"], c)
        r = fire()
        if _is_lag(r):                                   # read-after-write 500
            time.sleep(_RAW_SETTLE); r = fire()
        if (r.status_code == 404 and not spec["drop"]
                and spec["action"] in ("READ", "GET", "LIST")):   # entity-visibility lag
            time.sleep(_RAW_SETTLE); r = fire()
        if spec["action"] == "CREATE" and r.status_code < 400:
            created.ctx["made"] = c["entity"]            # record what the act created
        return r

    def result(name, resp, suite_ents, created):         # 3. assert (pure)
        c = suite_ents.ctx
        if resp is None:
            return "BLOCKED_OPA", c.get("granted", []), c.get("invalid", [])
        return classify(resp), c.get("granted", []), c.get("invalid", [])

    try:
        outcome, granted, invalid = run_case(action_id, suite, execute, result, root=tok)
        detail = ""
    except Exception as e:
        outcome, granted, invalid, detail = "ERROR", [], [], f"{type(e).__name__}: {str(e)[:110]}"
    if verbose or outcome in ("ERROR", "BLOCKED_OPA", "BLOCKED_CONFIG"):
        print(f"     ↳ {LABEL[action_id]}/{','.join(privs)} → {outcome}: {detail}")
    return outcome, granted, invalid

print(f"✅ {len(ACTIONS)} actions registered; run_case orchestrator ready "
      f"(Suite=build_suite · Execute=challenge · Result=classify · Cleaner=clean)")

✅ 11 actions registered; run_case orchestrator ready (Suite=build_suite · Execute=challenge · Result=classify · Cleaner=clean)


In [54]:
# ── ROLE PROFILES (footprint + expected allow/deny per doc §3) ───────────────
# allow = action_ids the role MUST be able to perform (expect AUTHORIZED)
# deny  = action_ids the role MUST be blocked from   (expect BLOCKED_PRIV)
# allow/deny encode the MEASURED boundary for each footprint (doc-privilege-results.md
# §4, 2026-07-02) — so a passing run confirms the build still behaves this way. Where
# the measured boundary differs from the role's *intent*, the fix is a footprint change
# (results §7), noted inline; the expectations here track reality, not the ideal.
ROLES = {
  "A · Data Engineer": dict(
      footprint=["NAMESPACE_CREATE","TABLE_CREATE","TABLE_WRITE_DATA","TABLE_READ_DATA","VIEW_CREATE"],
      allow=["ns_create","tbl_create","tbl_commit","tbl_read","view_create"],
      deny=["tbl_drop","ns_drop","view_drop"]),          # matches intent (8/8)
  "B · BI Analyst": dict(
      footprint=["TABLE_LIST","TABLE_READ_DATA","VIEW_LIST","VIEW_CREATE"],
      # MEASURED: VIEW_LIST does NOT authorize GET /views/{view} (→403), so view_get is a
      # boundary here — not a capability. Intent fix: add VIEW_READ_PROPERTIES (results §7).
      allow=["tbl_list","tbl_read","view_create"],
      deny=["tbl_create","tbl_commit","tbl_drop","view_get"]),
  "C · Auditor": dict(
      footprint=["CATALOG_MANAGE_METADATA"],
      # MEASURED: CATALOG_MANAGE_METADATA is NOT read-only — it authorizes every mutation,
      # so this footprint has NO deny boundary. Intent fix: use the granular read set for a
      # real read-only auditor (results §7).
      allow=["ns_list","tbl_list","tbl_read","view_get",
             "ns_create","ns_drop","tbl_create","tbl_drop","view_create","view_drop"],
      deny=[]),
  "D · Tenant Admin": dict(
      footprint=["CATALOG_MANAGE_CONTENT"],
      allow=["ns_create","ns_drop","tbl_create","tbl_drop","view_create","view_drop"],
      deny=[]),   # D's boundary is cross-catalog / mgmt-layer — tested separately below
}
print(f"{len(ROLES)} role profiles; "
      f"{sum(len(r['allow'])+len(r['deny']) for r in ROLES.values())} allow/deny assertions")

4 role profiles; 31 allow/deny assertions


In [55]:
# ── evaluate each role: provision the FULL footprint, assert allow & deny ────
ROLE_RESULTS = {}   # role -> dict(action_id -> (kind, outcome, ok))
ROLE_VOCAB   = {}   # role -> (granted, invalid)
print("Evaluating role profiles...\n")
for role, spec in ROLES.items():
    print(f"━━ {role}  footprint={spec['footprint']} ━━")
    res = {}
    granted_seen, invalid_seen = set(), set()
    for kind, aids, expected in [("allow", spec["allow"], "AUTHORIZED"),
                                 ("deny",  spec["deny"],  "BLOCKED_PRIV")]:
        for aid in aids:
            outcome, granted, invalid = probe(aid, spec["footprint"])
            granted_seen |= set(granted); invalid_seen |= set(invalid)
            ok = (outcome == expected)
            res[aid] = (kind, outcome, ok)
            mark = "✓" if ok else "✗"
            print(f"  [{kind:5}] {LABEL[aid]:16} → {outcome:14} expect {expected:13} {mark}")
    ROLE_RESULTS[role] = res
    ROLE_VOCAB[role]   = (sorted(granted_seen), sorted(invalid_seen))
    if invalid_seen:
        print(f"  ∅ footprint names rejected at grant (GRANT_INVALID): {sorted(invalid_seen)}")
    print()
print("✅ role evaluation complete")

Evaluating role profiles...

━━ A · Data Engineer  footprint=['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_WRITE_DATA', 'TABLE_READ_DATA', 'VIEW_CREATE'] ━━
  [allow] Namespace.CREATE → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.CREATE     → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.COMMIT     → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.READ       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] View.CREATE      → AUTHORIZED     expect AUTHORIZED    ✓
  [deny ] Table.DROP       → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓
  [deny ] Namespace.DROP   → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓
  [deny ] View.DROP        → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓

━━ B · BI Analyst  footprint=['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_LIST', 'VIEW_CREATE'] ━━
  [allow] Table.LIST       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.READ       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] View.CREATE      → AUTHORIZED     expect AUTHORIZED    ✓
  [deny ] Tab

In [56]:
# ── D · Tenant Admin boundary: cross-catalog isolation (via run_case) ────────
# A CATALOG_MANAGE_CONTENT holder on catalog X must NOT act on catalog Y. This
# case needs a custom two-catalog Suite, so it supplies its own suite_fn — the
# same run_case orchestrator still guarantees cleanup of BOTH catalogs + worker.
print("Tenant-Admin cross-catalog isolation test...\n")

def _xcat_suite(ents, root):                          # arrange: two catalogs
    cat_x = f"tenantA-{uuid.uuid4().hex[:6]}"
    cat_y = f"tenantB-{uuid.uuid4().hex[:6]}"
    create_catalog(cat_x, token=root);  ents.catalogs.append(cat_x)
    create_catalog(cat_y, token=root, properties={
        "polaris.config.drop-with-purge.enabled":"true",
        "polaris.config.purge-view-metadata-on-drop":"false"}); ents.catalogs.append(cat_y)
    ent = f"e{uuid.uuid4().hex[:6]}"
    # build a real table in cat_y with a full scaff principal, then destroy scaff
    shdr, sn = instance_principal(cat_y, privileges=["CATALOG_MANAGE_CONTENT"], suffix="scaff", root=root)
    ents.principals.append(sn[0]); ents.principal_roles.append(sn[1]); ents.catalog_roles.append((cat_y, sn[2]))
    _mk_ns(cat_y, shdr); _mk_tbl(cat_y, shdr, {"entity": ent})
    delete_instance_principal(sn, catalog=cat_y, root=root)
    # worker holds CATALOG_MANAGE_CONTENT on cat_x ONLY
    whdr, g, inv, wn = provision_worker_multi(cat_x, ["CATALOG_MANAGE_CONTENT"], root=root)
    ents.principals.append(wn[0]); ents.principal_roles.append(wn[1]); ents.catalog_roles.append((cat_x, wn[2]))
    ents.ctx.update(cat_x=cat_x, cat_y=cat_y, entity=ent, worker_hdr=whdr)

def _xcat_execute(ents, created, root):               # act: cat_x admin drops in cat_y
    c = ents.ctx
    return requests.delete(
        f"{BASE_CAT}/{c['cat_y']}/namespaces/{NS(c['cat_y'])}/tables/{c['entity']}",
        headers=c["worker_hdr"])

def _xcat_result(name, resp, ents, created):          # assert (pure)
    x = classify(resp)
    print(f"  cat_x admin → DROP table in cat_y → [{resp.status_code}] {x}")
    return x

xresult = run_case("xcat_isolation", _xcat_suite, _xcat_execute, _xcat_result, root=tok)
isolated = xresult in ("BLOCKED_PRIV","BLOCKED_OPA","NOT_FOUND_404")
print(f"\n{'✅' if isolated else '⚠️'} cross-catalog isolation "
      f"{'CONFIRMED' if isolated else 'NOT confirmed'}: a tenant admin "
      f"{'cannot' if isolated else 'COULD'} reach a sibling catalog.")
ROLE_RESULTS["D · Tenant Admin"]["xcat_drop"] = ("deny", xresult, isolated)

Tenant-Admin cross-catalog isolation test...

  cat_x admin → DROP table in cat_y → [403] BLOCKED_PRIV

✅ cross-catalog isolation CONFIRMED: a tenant admin cannot reach a sibling catalog.


In [57]:
# ── per-role PASS/FAIL cards ─────────────────────────────────────────────────
# Primary output is PLAIN TEXT (always visible in any frontend). A rich Markdown
# render is attempted as a bonus where the frontend supports text/markdown.
SYM = {"AUTHORIZED":"✅","BLOCKED_PRIV":"🔒","GRANT_INVALID":"∅","BLOCKED_OPA":"⚠️OPA",
       "BLOCKED_CONFIG":"⚠️CFG","LAG_500":"…","NOT_FOUND_404":"404","ERROR":"💥"}
# 1) guaranteed-visible plain-text report ------------------------------------
print("=" * 66)
print("USE-CASE ROLE BOUNDARY REPORT")
print("=" * 66)
for role, res in ROLE_RESULTS.items():
    granted, invalid = ROLE_VOCAB.get(role, ([],[]))
    npass = sum(1 for _,_,ok in res.values() if ok)
    ntot  = len(res)
    verdict = "PASS ✅" if npass == ntot else f"REVIEW ⚠️  ({npass}/{ntot})"
    print(f"\n#### {role} — {verdict}")
    print(f"     footprint granted: {granted}")
    if invalid:
        print(f"     ∅ GRANT_INVALID (build rejected): {invalid} → role only partially")
        print(f"       provisioned; a deny-side BLOCKED_PRIV may be a vocabulary gap.")
    print(f"     {'ACTION':18} {'SIDE':6} {'MEASURED':16} OK")
    print(f"     {'-'*18} {'-'*6} {'-'*16} --")
    for aid,(kind,outcome,ok) in res.items():
        lab = LABEL.get(aid, aid)
        print(f"     {lab:18} {kind:6} {SYM.get(outcome,''):2}{outcome:14} {'✓' if ok else '✗'}")
print("\nLegend: ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · 💥 ERROR · "
      "OK ✓ = measured matches the role's intended boundary.")

# 2) bonus: rich Markdown render (ignored silently if frontend can't render) ---
try:
    from IPython.display import Markdown, display
    blocks = []
    for role, res in ROLE_RESULTS.items():
        granted, invalid = ROLE_VOCAB.get(role, ([],[]))
        npass = sum(1 for _,_,ok in res.values() if ok); ntot = len(res)
        verdict = "✅ PASS" if npass == ntot else f"⚠️ {npass}/{ntot}"
        rows = [f"| {LABEL.get(aid,aid)} | {kind} | {SYM.get(outcome,outcome)} {outcome} | {'✓' if ok else '✗'} |"
                for aid,(kind,outcome,ok) in res.items()]
        inv = (f"\n\n> ∅ **GRANT_INVALID footprint names** (build rejected): `{invalid}` — role only "
               f"partially provisioned; any `deny`-side BLOCKED_PRIV may be a vocabulary gap.") if invalid else ""
        blocks.append(f"#### {role} — {verdict}\n\nFootprint granted: `{granted}`{inv}\n\n"
                      f"| Action | Side | Measured | OK |\n|---|---|---|---|\n" + "\n".join(rows))
    display(Markdown("### Use-Case Role Boundary Report\n\n" + "\n\n---\n\n".join(blocks)))
except Exception:
    pass   # plain-text report above is the source of truth

USE-CASE ROLE BOUNDARY REPORT

#### A · Data Engineer — PASS ✅
     footprint granted: ['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE']
     ACTION             SIDE   MEASURED         OK
     ------------------ ------ ---------------- --
     Namespace.CREATE   allow  ✅ AUTHORIZED     ✓
     Table.CREATE       allow  ✅ AUTHORIZED     ✓
     Table.COMMIT       allow  ✅ AUTHORIZED     ✓
     Table.READ         allow  ✅ AUTHORIZED     ✓
     View.CREATE        allow  ✅ AUTHORIZED     ✓
     Table.DROP         deny   🔒 BLOCKED_PRIV   ✓
     Namespace.DROP     deny   🔒 BLOCKED_PRIV   ✓
     View.DROP          deny   🔒 BLOCKED_PRIV   ✓

#### B · BI Analyst — PASS ✅
     footprint granted: ['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_CREATE', 'VIEW_LIST']
     ACTION             SIDE   MEASURED         OK
     ------------------ ------ ---------------- --
     Table.LIST         allow  ✅ AUTHORIZED     ✓
     Table.READ         allow  ✅ AUTHORIZED     ✓
  

### Use-Case Role Boundary Report

#### A · Data Engineer — ✅ PASS

Footprint granted: `['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.COMMIT | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |
| Namespace.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |
| View.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |

---

#### B · BI Analyst — ✅ PASS

Footprint granted: `['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_CREATE', 'VIEW_LIST']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Table.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | deny | 🔒 BLOCKED_PRIV | ✓ |
| Table.COMMIT | deny | 🔒 BLOCKED_PRIV | ✓ |
| Table.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |
| View.GET | deny | 🔒 BLOCKED_PRIV | ✓ |

---

#### C · Auditor — ✅ PASS

Footprint granted: `['CATALOG_MANAGE_METADATA']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.GET | allow | ✅ AUTHORIZED | ✓ |
| Namespace.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Namespace.DROP | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.DROP | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| View.DROP | allow | ✅ AUTHORIZED | ✓ |

---

#### D · Tenant Admin — ✅ PASS

Footprint granted: `['CATALOG_MANAGE_CONTENT']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Namespace.DROP | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.DROP | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| View.DROP | allow | ✅ AUTHORIZED | ✓ |
| xcat_drop | deny | 🔒 BLOCKED_PRIV | ✓ |

In [58]:
# ── SUMMARY: overall role pass/fail + discovered vocabulary ──────────────────
print("ROLE BOUNDARY VERDICT:\n")
all_ok = True
for role, res in ROLE_RESULTS.items():
    npass = sum(1 for _,_,ok in res.values() if ok)
    ntot  = len(res)
    all_ok &= (npass == ntot)
    print(f"  {role:20} {npass}/{ntot} assertions held "
          f"{'✅' if npass==ntot else '⚠️ review mismatches'}")

valid   = sorted({p for g,_ in ROLE_VOCAB.values() for p in g})
invalid = sorted({p for _,i in ROLE_VOCAB.values() for p in i})
print("\nDISCOVERED PRIVILEGE VOCABULARY (across all role footprints):")
print(f"  ✅ accepted at grant time : {valid}")
print(f"  ∅ rejected (GRANT_INVALID): {invalid}")
print(f"\n{'✅ ALL ROLES MATCH BASELINE' if all_ok else '⚠️ DEVIATION FROM BASELINE'} — "
      "expectations track the measured 2026-07-02 behavior (doc-privilege-results.md §4); "
      "a deviation here means the build changed. Intended-boundary fixes: results §7.")

ROLE BOUNDARY VERDICT:

  A · Data Engineer    8/8 assertions held ✅
  B · BI Analyst       7/7 assertions held ✅
  C · Auditor          10/10 assertions held ✅
  D · Tenant Admin     7/7 assertions held ✅

DISCOVERED PRIVILEGE VOCABULARY (across all role footprints):
  ✅ accepted at grant time : ['CATALOG_MANAGE_CONTENT', 'CATALOG_MANAGE_METADATA', 'NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_LIST', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE', 'VIEW_LIST']
  ∅ rejected (GRANT_INVALID): []

✅ ALL ROLES MATCH BASELINE — expectations track the measured 2026-07-02 behavior (doc-privilege-results.md §4); a deviation here means the build changed. Intended-boundary fixes: results §7.


In [59]:
# ── teardown sweep: reclaim stragglers (this run + any prior-run leftovers) ───
# Every probe tears down its own catalog + scaff + worker. This belt-and-suspenders
# pass removes anything left behind — including principals from the OLD '-role-'
# naming and any test catalog whose teardown was interrupted — so no residue can
# affect other tests.
TEST_PREFIXES = ["hzr-", "tenantA-", "tenantB-"]

# 1) leftover test catalogs → recursive teardown (drops ns/tables/views + purge)
_cats = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok))
_names = [(c.get("name") if isinstance(c, dict) else c)
          for c in (_cats.json().get("catalogs", []) if _cats.status_code == 200 else [])]
_left = [n for n in _names if n and any(n.startswith(p) for p in TEST_PREFIXES)]
for n in _left:
    try: teardown_catalog(n, root=tok)
    except Exception as e: print(f"  ⚠️ teardown_catalog({n}): {e}")

# 2) leftover principals / principal-roles (covers current 'worker'/'scaff' AND
#    the deprecated 'role' suffix from earlier runs)
_swept = sweep_instance_principals(
    prefixes=TEST_PREFIXES,
    suffixes=("scaff", "worker", "role", "inst", "full", "scaffold"))

print(f"🧹 cleanup: tore down {len(_left)} leftover test catalog(s); "
      f"swept {_swept} principal/role object(s). Polaris is clean.")

🧹 swept 0 leftover instance principal/role objects
🧹 cleanup: tore down 0 leftover test catalog(s); swept 0 principal/role object(s). Polaris is clean.
